# ಪಾಠ 11 - ಮಾದರಿ ಸನ್ನಿವೇಶ ಪ್ರೋಟೋಕಾಲ್ (MCP)

The **ಮಾದರಿ ಸನ್ನಿವೇಶ ಪ್ರೋಟೋಕಾಲ್ (MCP)** ಒಂದು ತೆರೆಯಾದ ಮಾನದಂಡವಾಗಿದೆ, ಇದು ಏಜೆಂಟ್‌ಗಳಿಗೆ ಚಾಲನೆಯ ಸಮಯದಲ್ಲಿ ಸಾಧನಗಳು, ಸಂಪನ್ಮೂಲಗಳು ಮತ್ತು ಡೇಟಾ ಮೂಲಗಳನ್ನು ಡೈನಾಮಿಕ್‌ವಾಗಿ ಕಂಡುಹಿಡಿದು ಬಳಸಲು ಅನುಮತಿಸುತ್ತದೆ. ಒಂದು ಏಜೆಂಟ್‌ಗೆ ಸಾಧನಗಳನ್ನು ಹಾರ್ಡ್‌ಕೋಡ್ ಮಾಡುವ ಬದಲು, MCP ಏಜೆಂಟ್‌들에게 ಬೇಡಿಕೆಯಂತೆ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಪ್ರadarಶಿಸುವ ಹೊರಗಿನ ಸರ್ವರ್‌ಗಳಿಗೆ ಸಂಪರ್ಕ ಹೊಂದಲು ಅವಕಾಶ ನೀಡುತ್ತದೆ.

In this lesson, you'll learn:
- MCP ಏನು ಮತ್ತು ಏಕೆ ಇದು ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳಿಗೆ ಮಹತ್ವದಾಗಿದೆ
- MCP ಕ್ಲೈಂಟ್-ಸರ್ವರ್ ವ್ಯವಸ್ಥೆ ಹೇಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ
- MCP-ಶೈಲಿಯ ಸಾಧನ ಪತ್ತೆಯನ್ನು ಬಳಸುವ ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸುವುದು


## ಸೆಟ್‌ಅಪ್

**ಪೂರ್ವಾವಶ್ಯಕತೆಗಳು:**
- ಒಂದು ನಿಯೋಜಿತ ಮಾದರಿಯನ್ನು ಹೊಂದಿರುವ Azure AI Foundry ಪ್ರಾಜೆಕ್ಟ್
- ಪ್ರಮಾಣೀಕರಣಕ್ಕಾಗಿ `az login` ಅನ್ನು ಓಡಿಸಿ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) ಎಂಬುದು ಏನು?

MCP ಎಐ ಏಜೆಂಟ್‌ಗಳು ಹೊರಗಿನ ಉಪಕರಣಗಳು ಮತ್ತು ಡೇಟಾ ಮೂಲಗಳನ್ನು ಪತ್ತೆಮಾಡಿ ಅವುಗಳೊಂದಿಗೆ ಪರಸ್ಪರ ಕ್ರಿಯೆಗೊಳಿಸಲು ಒಂದು ಪ್ರಮಾಣಿತ ವಿಧಾನವನ್ನು ನಿರ್ಧರಿಸುತ್ತದೆ:

- **MCP Server**: ಮಾನಕ ಪ್ರೋಟೋಕಾಲ್ ಮೂಲಕ ಉಪಕರಣಗಳು, ಸಂಪನ್ಮೂಲಗಳು ಮತ್ತು ಪ್ರಾಂಪ್ಟ್‌ಗಳನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ
- **MCP Client**: ಸರ್ವರ್‌ಗಳಿಗೆ ಸಂಪರ್ಕ ಹೊಂದಿ ಲಭ್ಯವಿರುವ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಕಂಡುಹಿಡಿಯುವ ಏಜೆಂಟ್ ರನ್‌ಟೈಮ್
- **ಡೈನಾಮಿಕ್ ಅನ್ವೇಷಣೆ**: ಏಜೆಂಟ್‌들에게 ಹಾರ್ಡ್‍ಕೋಡ್ ಮಾಡಿದ ಉಪಕರಣಗಳ ಅಗತ್ಯವಿಲ್ಲ — ಅವು ರನ್‌ಟೈಮ್‌ನಲ್ಲಿ ಲಭ್ಯವಿರುವುದನ್ನು ಕಂಡುಹಿಡಿಯುತ್ತವೆ

ಇದು ವಿಸ್ತರಣೀಯ ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳನ್ನು ನಿರ್ಮಿಸಲು ಪರಿಣಾಮಕಾರಿಯಾಗಿದೆ, ಅಲ್ಲಿ ಹೊಸ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಏಜೆಂಟ್ ಕೋಡ್ ಬದಲಾಯಿಸದೆ ಸೇರಿಸಬಹುದು.


## MCP ಹೇಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ಏಜೆಂಟ್ (MCP client) ಒಂದು MCP server ಗೆ ಸಂಪರ್ಕಿಸುತ್ತದೆ
2. ಸರ್ವರ್ ಲಭ್ಯವಿರುವ ಉಪಕರಣಗಳ ಮತ್ತು ಅವುಗಳ ಸ್ಕೀಮೆಗಳ ಪಟ್ಟಿಯೊಂದಿಗೆ ಪ್ರತಿಕ್ರಿಯಿಸುತ್ತದೆ
3. ನಂತರ ಏಜೆಂಟ್ ತನ್ನ ನಿರ್ಣಯಿಸುವ ಪ್ರಕ್ರಿಯೆಯ ವೇಳೆ ಕಂಡುಹಿಡಿದ ಯಾವುದೇ ಉಪಕರಣವನ್ನು ಕರೆಯಬಹುದು
4. ಫಲಿತಾಂಶಗಳು ಅದೇ ಪ್ರೋಟೋಕಾಲ್ ಮೂಲಕ ಹಿಂದಕ್ಕೆ ಹರಿಯುತ್ತವೆ


## MCP ಉಪಕರಣ ಪತ್ತೆ ಅನುಕರಣ

ನಿಜವಾದ MCP ಸರ್ವರ್‌ಗೆ ಸರ್ವರ್ ಪ್ರಕ್ರಿಯೆಯೊಂದು ಚಾಲನದಲ್ಲಿರಬೇಕಾಗುತ್ತದೆ, ಆದ್ದರಿಂದ ನಾವು `@tool` ಕಾರ್ಯಗಳನ್ನು ಬಳಸಿ MCP-ಸಂಪರ್ಕಿತ ವಸತಿ ಸೇವೆ ನೀಡುವುದು ಹೇಗೆ ಎಂದು ಅನುಕರಣ ಮಾಡುವ ಮಾದರಿಯನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತೇವೆ.

ಉತ್ಪಾದನಾ ಪರಿಸರದಲ್ಲಿ, ಈ ಉಪಕರಣಗಳನ್ನು ಸ್ಥಳೀಯವಾಗಿ ನಿರ್ಧರಿಸುವ ಬದಲು MCP ಸರ್ವರ್‌ನಿಂದ ಡೈನಾಮಿಕ್ ಆಗಿ ಪತ್ತೆಹಚ್ಚಲಾಗುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ಶೈಲಿಯ ಉಪಕರಣಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ನಿರ್ಮಿಸುವುದು


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ಉತ್ಪಾದನೆಯಲ್ಲಿ MCP

ಉತ್ಪಾದನಾ ಪರಿಸರದಲ್ಲಿ, MCP ಶಕ್ತಿಶಾಲಿ ಮಾದರಿಗಳನ್ನು ಸಕ್ರಿಯಗೊಳಿಸುತ್ತದೆ:

- **ಡೈನಾಮಿಕ್ ಉಪಕರಣ ಅನ್ವೇಷಣೆ**: ಏಜೆಂಟ್ಗಳು MCP ಸರ್ವರ್‌ಗಳಿಗೆ ಸಂಪರ್ಕಿಸಿ ರನ್‌ಟೈಮ್‌ನಲ್ಲಿ ಉಪಕರಣಗಳನ್ನು ಕಂಡುಹಿಡಿಯುತ್ತವೆ
- **ಪ್ರತ್ಯೇಕೀಕೃತ ವಾಸ್ತುಶಿಲ್ಪ**: ಉಪಕರಣ ಪೂರೈಕೆದಾರರು ಏಜೆಂಟ್‌ನಿಂದ ಸ್ವತಂತ್ರವಾಗಿ ನವೀಕರಿಸಬಹುದು
- **ಸಂಸ್ಥೆಗಳ ದಾಟಿ ಹಂಚಿಕೊಳ್ಳುವಿಕೆ**: ತಂಡಗಳು MCP ಸರ್ವರ್‌ಗಳ ಮೂಲಕ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಬಹಿರಂಗಪಡಿಸಬಹುದು ಮತ್ತು ಅವುಗಳನ್ನು ಯಾವುದೇ ಏಜೆಂಟ್ ಬಳಸಬಹುದು
- **Microsoft Agent Framework ಬೆಂಬಲ**: MAF ಒಳಗೊಳಿಸಿದ `mcp` ಇಂಟಿಗ್ರೇಶನ್ ಮೂಲಕ MCP ಕ್ಲೈಂಟ್ ಬೆಂಬಲವನ್ನು ಹೊಂದಿದೆ

To use a real MCP server with MAF, you would connect via `hosted_mcp_tool()` or the MCP client integration.

**ಇನ್ನಷ್ಟು ತಿಳಿಯಿರಿ:**
- [MCP ವಿಶೇಷಣೆ](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP ಬೆಂಬಲ](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಕಲಿತವು:
- **MCP** ಏಜೆಂಟ್‌ಗಳು ಮತ್ತು ಉಪಕರಣ ಒದಗಿಸುವವರ ನಡುವೆ ಗತಿಶೀಲ ಉಪಕರಣ ಪತ್ತೆಗಾಗಿ ಇರುವ ಓಪನ್ ಸ್ಟ್ಯಾಂಡರ್ಡ್ ಆಗಿದೆ
- **client-server architecture** ಏಜೆಂಟ್‌ಗಳಿಗೆ ಚಾಲನೆಯ ಸಮಯದಲ್ಲಿ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಕಂಡುಹಿಡಿಯಲು ಅವಕಾಶ ನೀಡುತ್ತದೆ
- MCP **ವಿಸ್ತರಣೀಯ, ವಿಭಜಿತ ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳನ್ನು** ಸಾಧ್ಯಮಾಡುತ್ತದೆ ಅಲ್ಲಿ ಉಪಕರಣಗಳನ್ನು ಕೋಡ್ ಬದಲಾವಣೆಗಳಿಲ್ಲದೆ ಸೇರಿಸಬಹುದು
- Microsoft Agent Framework ಉತ್ಪಾದನಾ ಬಳಕೆಗೆ **ಒಳಗೊಂಡಿರುವ MCP ಬೆಂಬಲ** ಒದಗಿಸುತ್ತದೆ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ಜವಾಬ್ದಾರಿಯ ನಿರಾಕರಣೆ:
ಈ ದಾಖಲೆ AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಗೆ ಪ್ರಯತ್ನಿಸಿದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳಿರಬಹುದೆಂದು ದಯವಿಟ್ಟು ಗಮನದಲ್ಲಿಡಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಾಖಲೆಗಳನ್ನು ಪ್ರಾಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಗಂಭೀರ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ಅಸಮಂಜಸತೆಗಳು ಅಥವಾ ತಪ್ಪು ಅರ್ಥಗೊಳ್ಳುವಿಕೆಗೆ ನಾವು ಹೊಣೆಗಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
